In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os,sys
import time
import gc
import pickle
from datetime import datetime
import cmaps
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
# 配置参数
CACHE_DIR = "../cache/kelvin_wave/"
REGRESSION_CACHE_DIR = os.path.join(CACHE_DIR, "pr_regression_composite")
FIG_SAVE_DIR = "../figures/vertical_profile_composite/"
from pathlib import Path
WAVE_TOOLS_PATH = Path("/work/mh1498/m301257/wave_tools/")
sys.path.insert(0, str(WAVE_TOOLS_PATH.parent))
from wave_tools.utils import create_colormap
mycmap = create_colormap() 


# 创建必要的目录
for d in [CACHE_DIR, REGRESSION_CACHE_DIR, FIG_SAVE_DIR]:
    os.makedirs(d, exist_ok=True)


EXPERIMENTS = ['CNTL', 'P4K', '4CO2']


In [2]:

def load_filtered_data(var_name, exp_list=None):
    """
    加载滤波后的数据
    
    Parameters:
    -----------
    var_name : str
        变量名称 ('pr', 'ua', 'va', 'omega', 'hus')
    exp_list : list, optional
        实验列表
    
    Returns:
    --------
    data_dict : dict
        {exp_name: xr.DataArray}
    """
    if exp_list is None:
        exp_list = EXPERIMENTS
    
    data_dict = {}
    
    # 定义可能的缓存目录（按优先级顺序）
    possible_dirs = [
        CACHE_DIR,                                    # ./cache/kelvin_wave/
        "../cache/kelvin_wave_3d/",                    # 3D变量目录
        os.path.join(CACHE_DIR, "../kelvin_wave_3d/") # 相对路径
    ]
    
    print(f"\n📂 Loading {var_name} data...")
    for exp in exp_list:
        loaded = False
        
        # 尝试在不同目录中查找文件
        for cache_dir in possible_dirs:
            cache_file = os.path.join(cache_dir, f'kelvin_{var_name}_{exp.lower()}.nc')
            
            if os.path.exists(cache_file):
                try:
                    # 根据变量类型选择合适的chunk大小
                    if var_name in ['hus', 'ua', 'va', 'omega']:
                        # 3D变量：使用chunk加速
                        chunks = {'time': 500, 'lat': -1, 'lon': -1}
                    else:
                        # 2D变量
                        chunks = {'time': 1000, 'lat': -1, 'lon': -1}
                    
                    # 尝试作为Dataset加载（针对hus等新保存的数据）
                    try:
                        ds = xr.open_dataset(cache_file, chunks=chunks)
                        # 提取变量
                        if var_name in ds:
                            data = ds[var_name]
                        elif 'hus' in ds:
                            data = ds['hus']
                        else:
                            data = ds[list(ds.data_vars)[0]]
                    except:
                        # 如果失败，尝试作为DataArray加载（旧格式）
                        data = xr.open_dataarray(cache_file, chunks=chunks)
                    
                    # 标准化维度名称：将 'level' 重命名为 'lev'（如果存在）
                    if 'level' in data.dims:
                        data = data.rename({'level': 'lev'})
                    
                    data_dict[exp] = data
                    file_size = os.path.getsize(cache_file) / (1024**2)
                    print(f"  ✅ {exp}: Loaded from {os.path.basename(cache_dir)}, "
                          f"Shape={data.shape}, Size={file_size:.1f}MB")
                    if 'lev' in data.dims:
                        print(f"      Levels: {len(data.lev)}, Chunked: {data.chunks is not None}")
                    loaded = True
                    break
                except Exception as e:
                    print(f"  ❌ {exp}: Error loading from {cache_dir} - {str(e)}")
        
        if not loaded:
            print(f"  ⚠️  {exp}: Cache file not found in any directory!")
    
    return data_dict

print("✅ load_filtered_data 函数定义完成")

✅ load_filtered_data 函数定义完成




对hus场进行合成分析，使用与降水和风场相同的事件时间

In [3]:
def load_detected_events(composite_dir=None, verbose=True):
    """
    加载已保存的事件检测结果
    
    Parameters:
    -----------
    composite_dir : str, optional
        事件数据存储目录，默认为 './composite_data'
    verbose : bool
        是否打印详细信息
    
    Returns:
    --------
    all_events : dict
        {exp_name: {'event_dates': [...], 'event_intensities': [...], 'n_events': ...}}
    success : bool
        是否成功加载
    """
    import json
    import os
    
    if composite_dir is None:
        composite_dir = os.path.join(os.getcwd(), 'composite_data')
    
    events_file = "/work/mh1498/m301257/composite_data/detected_events.json"
    
    if verbose:
        print("\n" + "="*80)
        print("📂 Loading Detected Events")
        print("="*80)
        print(f"File: {events_file}")
        print()
    
    if not os.path.exists(events_file):
        if verbose:
            print(f"⚠️  Event file not found: {events_file}")
            print("   Please run the event detection and save step first.")
        return None, False
    
    try:
        with open(events_file, 'r') as f:
            all_events = json.load(f)
        
        if verbose:
            print("✅ Event detection results loaded successfully!")
            print(f"\n📊 Events summary:")
            for exp, events_info in all_events.items():
                print(f"   {exp}: {events_info['n_events']} events")
                print(f"      Event dates range: {min(events_info['event_dates'])} to {max(events_info['event_dates'])}")
        
        return all_events, True
        
    except Exception as e:
        if verbose:
            print(f"❌ Failed to load events: {e}")
        return None, False

def compute_composite_with_std(data, event_dates, lags):
    """
    计算复合分析及其标准差（用于显著性检验）
    优化版本：批量提取时间索引，减少I/O操作
    
    Returns:
    --------
    composite_mean : xr.DataArray
        复合平均场
    composite_std : xr.DataArray
        复合标准差
    n_events : int
        有效事件数
    """
    import time
    start_time = time.time()
    
    # 预先筛选有效事件
    valid_events = []
    for event_idx in event_dates:
        if event_idx + min(lags) >= 0 and event_idx + max(lags) < len(data.time):
            valid_events.append(event_idx)
    
    if len(valid_events) == 0:
        return None, None, 0
    
    print(f"  📊 Processing {len(valid_events)} valid events with {len(lags)} lags...")
    
    # 批量构建所有需要的时间索引
    all_time_indices = []
    for event_idx in valid_events:
        for lag in lags:
            all_time_indices.append(event_idx + lag)
    
    # 一次性提取所有需要的时间切片（大幅减少I/O）
    print(f"  ⏳ Extracting {len(all_time_indices)} time slices...")
    extract_start = time.time()
    all_data = data.isel(time=all_time_indices).compute()  # 强制计算并加载到内存
    print(f"     ✓ Extraction took {time.time() - extract_start:.1f}s")
    
    # 重组数据为 (event, lag, ...) 结构
    print(f"  ⏳ Reshaping data...")
    n_events = len(valid_events)
    n_lags = len(lags)
    
    composites = []
    idx = 0
    for i in range(n_events):
        event_data = []
        for j in range(n_lags):
            event_data.append(all_data.isel(time=idx))
            idx += 1
        event_composite = xr.concat(event_data, dim='lag')
        event_composite['lag'] = list(lags)
        composites.append(event_composite)
    
    # 堆叠并计算统计量
    print(f"  ⏳ Computing statistics...")
    all_composites = xr.concat(composites, dim='event')
    composite_mean = all_composites.mean(dim='event')
    composite_std = all_composites.std(dim='event')
    
    elapsed = time.time() - start_time
    print(f"  ✅ Composite computed in {elapsed:.1f}s")
    
    return composite_mean, composite_std, n_events


print("✅ Composite analysis functions defined")    

def apply_composite_to_other_variable(variable_data, event_dates, lags, 
                                     exp_name='', verbose=True):
    """
    使用已检测到的事件，对其他变量进行超前滞后合成分析
    
    Parameters:
    -----------
    variable_data : xr.DataArray
        要进行合成分析的变量数据 (time, lat, lon) 或 (time, lev, lat, lon)
    event_dates : list of int
        从降水分析中获得的事件时间索引
    lags : range or list
        lag 天数，应该与降水合成使用相同的范围
    exp_name : str
        实验名称（用于输出）
    verbose : bool
        是否打印详细信息
    
    Returns:
    --------
    composite_mean : xr.DataArray
        复合平均场
    composite_std : xr.DataArray
        复合标准差
    n_events : int
        有效事件数
    
    Example:
    --------
    # 加载事件信息
    all_events, success = load_detected_events()
    
    # 加载潜热通量数据
    lhf_data = xr.open_dataarray('latent_heat_flux_cntl.nc')
    
    # 使用降水事件对潜热通量进行合成
    lhf_composite, lhf_std, n = apply_composite_to_other_variable(
        lhf_data, 
        all_events['CNTL']['event_dates'], 
        range(-4, 5),
        exp_name='CNTL'
    )
    """
    if verbose:
        print(f"\n{'='*70}")
        print(f"🔄 Computing composite for {exp_name}")
        print(f"{'='*70}")
        print(f"   Variable shape: {variable_data.shape}")
        print(f"   Number of events: {len(event_dates)}")
        print(f"   Lag range: {min(lags)} to {max(lags)} days")
    
    # 使用已有的合成函数
    composite_mean, composite_std, n_events = compute_composite_with_std(
        data=variable_data,
        event_dates=event_dates,
        lags=lags
    )
    
    if verbose:
        if composite_mean is not None:
            print(f"✅ Composite computed successfully")
            print(f"   Valid events used: {n_events}")
            print(f"   Composite shape: {composite_mean.shape}")
            print(f"   Value range: [{composite_mean.min().values:.4e}, {composite_mean.max().values:.4e}]")
        else:
            print(f"❌ Failed to compute composite")
        print(f"{'='*70}")
    
    return composite_mean, composite_std, n_events



✅ Composite analysis functions defined


In [5]:
hus_data_dict = load_filtered_data('hus')

pr_data_dict = load_filtered_data('pr')


📂 Loading hus data...
  ✅ CNTL: Loaded from , Shape=(22, 5114, 15, 180), Size=1975.8MB
      Levels: 22, Chunked: True
  ✅ P4K: Loaded from , Shape=(22, 5114, 15, 180), Size=1974.5MB
      Levels: 22, Chunked: True
  ✅ 4CO2: Loaded from , Shape=(22, 5114, 15, 180), Size=1975.7MB
      Levels: 22, Chunked: True

📂 Loading pr data...
  ✅ CNTL: Loaded from , Shape=(5114, 15, 180), Size=90.7MB
  ✅ P4K: Loaded from , Shape=(5114, 15, 180), Size=90.8MB
  ✅ 4CO2: Loaded from , Shape=(5114, 15, 180), Size=90.7MB


/tmp/ipykernel_3769932/1053463096.py:49: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 500. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(cache_file, chunks=chunks)
/tmp/ipykernel_3769932/1053463096.py:49: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 500. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(cache_file, chunks=chunks)
/tmp/ipykernel_3769932/1053463096.py:49: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 500. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(cache_file, chunks=chunks)
/tmp/ipykernel_3769932/1053463096.py:49: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 1000. This could degrade performanc

In [ ]:
# 诊断：检查hus数据状态
print("="*70)
print("🔍 Diagnosing hus data")
print("="*70)

for exp in ['CNTL', 'P4K', '4CO2']:
    if exp in hus_data_dict:
        data = hus_data_dict[exp]
        print(f"\n📍 {exp}:")
        print(f"   Type: {type(data)}")
        print(f"   Shape: {data.shape}")
        print(f"   Dims: {data.dims}")
        print(f"   Dtype: {data.dtype}")
        print(f"   Chunked: {data.chunks is not None}")
        if data.chunks:
            print(f"   Chunks: {data.chunks}")
        
        # 估算内存大小
        nbytes = data.nbytes
        print(f"   Memory: {nbytes / 1e9:.2f} GB")
        
        # 检查是否是dask array
        import dask.array as da
        if isinstance(data.data, da.Array):
            print(f"   ✓ Using Dask (lazy evaluation)")
        else:
            print(f"   ⚠️  Data already in memory (not lazy)")
    else:
        print(f"\n❌ {exp}: Not loaded")

print("\n" + "="*70)

In [6]:
wa_data_dict = load_filtered_data('omega')


📂 Loading omega data...
  ✅ CNTL: Loaded from , Shape=(5114, 22, 15, 180), Size=2317.6MB
      Levels: 22, Chunked: True
  ✅ P4K: Loaded from , Shape=(5114, 22, 15, 180), Size=2317.6MB
      Levels: 22, Chunked: True
  ✅ 4CO2: Loaded from , Shape=(5114, 22, 15, 180), Size=2317.6MB
      Levels: 22, Chunked: True


In [7]:
pr_composites_dict = {}
pr_std_dict = {}
all_events, success = load_detected_events()    
for exp in EXPERIMENTS:
    pr_composite, pr_std, n_events = compute_composite_with_std(
        data=pr_data_dict[exp],
        event_dates=all_events[exp]['event_dates'],
        lags=range(-4, 5)
    )
    pr_composites_dict[exp] = pr_composite
    pr_std_dict[exp] = pr_std
    print(f"✅ {exp} precipitation composite computed with {n_events} events")


📂 Loading Detected Events
File: /work/mh1498/m301257/composite_data/detected_events.json

✅ Event detection results loaded successfully!

📊 Events summary:
   CNTL: 546 events
      Event dates range: 19 to 5095
   P4K: 548 events
      Event dates range: 12 to 5090
   4CO2: 552 events
      Event dates range: 10 to 5100
  📊 Processing 546 valid events with 9 lags...
  ⏳ Extracting 4914 time slices...
     ✓ Extraction took 1.4s
  ⏳ Reshaping data...
  ⏳ Computing statistics...
  ✅ Composite computed in 4.7s
✅ CNTL precipitation composite computed with 546 events
  📊 Processing 548 valid events with 9 lags...
  ⏳ Extracting 4932 time slices...
     ✓ Extraction took 0.9s
  ⏳ Reshaping data...
  ⏳ Computing statistics...
  ✅ Composite computed in 4.0s
✅ P4K precipitation composite computed with 548 events
  📊 Processing 552 valid events with 9 lags...
  ⏳ Extracting 4968 time slices...
     ✓ Extraction took 0.7s
  ⏳ Reshaping data...
  ⏳ Computing statistics...
  ✅ Composite computed 

In [ ]:
def get_comosite_hus():
    """计算hus的合成分析（优化版本）"""
    import time
    
    print("\n" + "="*70)
    print("🌊 Computing Specific Humidity (hus) Composites")
    print("="*70)
    
    try:
        # 加载事件信息
        all_events, success = load_detected_events(verbose=False)
        if not success:
            print("❌ Failed to load events")
            return None
        
        hus_composite_dict = {}
        total_start = time.time()
        
        for exp_idx, exp in enumerate(['CNTL', 'P4K', '4CO2'], 1):
            print(f"\n📍 Processing {exp} ({exp_idx}/3)")
            print(f"   Events: {len(all_events[exp]['event_dates'])}")
            print(f"   Data shape: {hus_data_dict[exp].shape}")
            print(f"   Data dims: {hus_data_dict[exp].dims}")
            
            exp_start = time.time()
            
            hus_composite, hus_std, n_events = apply_composite_to_other_variable(
                hus_data_dict[exp],
                all_events[exp]['event_dates'],
                range(-4, 5),
                exp_name=exp,
                verbose=False
            )
            
            hus_composite_dict[exp] = hus_composite
            
            exp_elapsed = time.time() - exp_start
            print(f"   ✅ {exp} completed in {exp_elapsed:.1f}s ({exp_elapsed/60:.1f}min)")
            print(f"   Composite shape: {hus_composite.shape}")
            
            # 清理内存
            import gc
            gc.collect()
        

         
        return hus_composite_dict
        
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        import traceback
        traceback.print_exc()
        print("\n⚠️  Make sure hus data is loaded:")
        print("   hus_data_dict = load_filtered_data('hus')")
        return None

# hus_composite_dict = get_comosite_hus()

In [ ]:
def get_comosite_wa():
    """计算hus的合成分析（优化版本）"""

    try:
        # 加载事件信息
        all_events, success = load_detected_events(verbose=False)
        if not success:
            print("❌ Failed to load events")
            return None
        
        wa_composite_dict = {}
 
        
        for exp_idx, exp in enumerate(['CNTL', 'P4K', '4CO2'], 1):
            print(f"\n📍 Processing {exp} ({exp_idx}/3)")
            print(f"   Events: {len(all_events[exp]['event_dates'])}")
            print(f"   Data shape: {hus_data_dict[exp].shape}")
            print(f"   Data dims: {hus_data_dict[exp].dims}")
            
    
            wa_composite, wa_std, n_events = apply_composite_to_other_variable(
                wa_data_dict[exp],
                all_events[exp]['event_dates'],
                range(-4, 5),
                exp_name=exp,
                verbose=False
            )
            
            wa_composite_dict[exp] = wa_composite

            print(f"   Composite shape: {wa_composite.shape}")
            
            # 清理内存
            import gc
            gc.collect()
        

         
        return wa_composite_dict
        
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        import traceback
        traceback.print_exc()
        print("\n⚠️  Make sure hus data is loaded:")
        print("   hus_data_dict = load_filtered_data('hus')")
        return None

# hus_composite_dict = get_comosite_hus()

In [9]:
wa_composite_dict = get_comosite_wa()


🌊 Computing Specific Humidity (hus) Composites

📍 Processing CNTL (1/3)
   Events: 546
   Data shape: (22, 5114, 15, 180)
   Data dims: ('lev', 'time', 'lat', 'lon')
  📊 Processing 546 valid events with 9 lags...
  ⏳ Extracting 4914 time slices...
     ✓ Extraction took 4.6s
  ⏳ Reshaping data...
  ⏳ Computing statistics...
  ✅ Composite computed in 13.0s
   Composite shape: (9, 22, 15, 180)

📍 Processing P4K (2/3)
   Events: 548
   Data shape: (22, 5114, 15, 180)
   Data dims: ('lev', 'time', 'lat', 'lon')
  📊 Processing 548 valid events with 9 lags...
  ⏳ Extracting 4932 time slices...
     ✓ Extraction took 4.2s
  ⏳ Reshaping data...
  ⏳ Computing statistics...
  ✅ Composite computed in 12.1s
   Composite shape: (9, 22, 15, 180)

📍 Processing 4CO2 (3/3)
   Events: 552
   Data shape: (22, 5114, 15, 180)
   Data dims: ('lev', 'time', 'lat', 'lon')
  📊 Processing 552 valid events with 9 lags...
  ⏳ Extracting 4968 time slices...
     ✓ Extraction took 4.1s
  ⏳ Reshaping data...
  ⏳ Co

6.2.1 加载位势高度数据

In [10]:
# 1. 加载 zg 数据（位势高度，单位：m）
print("读取 zg 数据...")
zg_cntl = xr.open_dataset('/work/mh1498/m301257/3D_data/zg_2deg_interp_cntl.nc')['zg']
zg_p4k = xr.open_dataset('/work/mh1498/m301257/3D_data/zg_2deg_interp_p4k.nc')['zg']
zg_4co2 = xr.open_dataset('/work/mh1498/m301257/3D_data/zg_2deg_interp_4co2.nc')['zg']

# 2. 检查维度名称并标准化
print(f"  zg_cntl dimensions: {zg_cntl.dims}")
if 'level_full' in zg_cntl.dims:
    zg_cntl = zg_cntl.rename({'level_full': 'lev'})
    zg_p4k = zg_p4k.rename({'level_full': 'lev'})
    zg_4co2 = zg_4co2.rename({'level_full': 'lev'})
    print("  ✓ 维度 'level_full' 已重命名为 'lev'")
    
def _ocean(ds):
    fraction = xr.open_dataset(os.path.join("../processed_data", "ocean_mask_2deg.nc"))['__xarray_dataarray_variable__']
    return fraction == 1

# 4. 应用海洋 mask
zg_cntl_ocean = zg_cntl.where(_ocean)
zg_p4k_ocean = zg_p4k.where(_ocean)
zg_4co2_ocean = zg_4co2.where(_ocean)

# 5. 计算海洋平均高度（对每一层）
# zg 不随时间变化，所以只对空间平均
zg_cntl_mean = zg_cntl_ocean.mean(dim=['lat', 'lon'], skipna=True)
zg_p4k_mean = zg_p4k_ocean.mean(dim=['lat', 'lon'], skipna=True)
zg_4co2_mean = zg_4co2_ocean.mean(dim=['lat', 'lon'], skipna=True)

# 6. 转换为 km 单位
zg_cntl_km = zg_cntl_mean / 1000.0
zg_p4k_km = zg_p4k_mean / 1000.0
zg_4co2_km = zg_4co2_mean / 1000.0

# 7. 创建高度字典
zg_dict = {
    'CNTL': zg_cntl_km,
    'P4K': zg_p4k_km,
    '4CO2': zg_4co2_km
}

print(f"\n✓ zg 数据加载完成")
print(f"  高度范围: {zg_cntl_km.min().values:.2f} - {zg_cntl_km.max().values:.2f} km")
print(f"  层数: {len(zg_cntl_km)}")
print(f"  维度: {zg_cntl_km.dims}")

读取 zg 数据...
  zg_cntl dimensions: ('level_full', 'lat', 'lon')
  ✓ 维度 'level_full' 已重命名为 'lev'

✓ zg 数据加载完成
  高度范围: 0.01 - 22.86 km
  层数: 22
  维度: ('lev',)


## 将hus合成数据的lev坐标替换为高度坐标(km)

In [11]:
def assign_height_coordinate(composite_dict, zg_dict):
    """
    将合成数据的lev坐标替换为实际高度(km)
    
    Parameters:
    -----------
    composite_dict : dict
        {exp: xr.DataArray} 包含lev维度的合成数据
    zg_dict : dict
        {exp: xr.DataArray} 位势高度数据 (lev维度，单位km)
    
    Returns:
    --------
    composite_with_height : dict
        {exp: xr.DataArray} lev坐标已替换为height(km)
    """
    print("\n" + "="*70)
    print("🔧 Assigning height coordinates to hus composite data")
    print("="*70)
    
    composite_with_height = {}
    
    for exp in composite_dict.keys():
        print(f"\n📍 Processing {exp}...")
        
        hus_comp = composite_dict[exp]
        zg = zg_dict[exp]
        
        # 确保lev维度一致
        if 'lev' not in hus_comp.dims:
            print(f"   ⚠️  No 'lev' dimension found, skipping...")
            composite_with_height[exp] = hus_comp
            continue
        
        # 检查lev值是否匹配
        if not all(hus_comp.lev.values == zg.lev.values):
            print(f"   ⚠️  Warning: lev values don't match exactly")
            print(f"      hus lev: {hus_comp.lev.values[:3]}...")
            print(f"      zg lev: {zg.lev.values[:3]}...")
        
        # 提取高度值
        height_km = zg.values  # 已经是km单位
        
        # 创建新的DataArray，用height替换lev
        # 保持原有的其他坐标和维度
        dims = list(hus_comp.dims)
        lev_idx = dims.index('lev')
        dims[lev_idx] = 'height'
        
        # 构建新的坐标
        new_coords = {}
        for coord_name, coord_values in hus_comp.coords.items():
            if coord_name == 'lev':
                new_coords['height'] = ('height', height_km, {'units': 'km', 'long_name': 'Height above sea level'})
            else:
                new_coords[coord_name] = coord_values
        
        # 创建新的DataArray
        hus_with_height = xr.DataArray(
            data=hus_comp.values,
            dims=dims,
            coords=new_coords,
            attrs=hus_comp.attrs
        )
        
        # 添加描述性属性
        hus_with_height.attrs['height_coordinate'] = 'Converted from pressure levels using geopotential height'
        hus_with_height.attrs['height_units'] = 'km'
        
        composite_with_height[exp] = hus_with_height
        
        print(f"   ✅ Height coordinate assigned")
        print(f"      New dims: {hus_with_height.dims}")
        print(f"      Height range: {height_km.min():.2f} - {height_km.max():.2f} km")
        print(f"      Shape: {hus_with_height.shape}")
    
    print("\n" + "="*70)
    print("✅ All experiments processed")
    print("="*70)
    
    return composite_with_height

# # 执行坐标转换
# hus_composite_with_height = assign_height_coordinate(hus_composite_dict, zg_dict)

# # 检查结果
# print("\n📊 Verification:")
# for exp in ['CNTL', 'P4K', '4CO2']:
#     if exp in hus_composite_with_height:
#         data = hus_composite_with_height[exp]
#         print(f"\n{exp}:")
#         print(f"  Dims: {data.dims}")
#         print(f"  Coords: {list(data.coords.keys())}")
#         if 'height' in data.coords:
#             print(f"  Height values (first 5): {data.height.values[:5]}")

In [12]:
# 执行坐标转换
wa_composite_with_height = assign_height_coordinate(wa_composite_dict, zg_dict)

# 检查结果
print("\n📊 Verification:")
for exp in ['CNTL', 'P4K', '4CO2']:
    if exp in wa_composite_with_height:
        data = wa_composite_with_height[exp]
        print(f"\n{exp}:")
        print(f"  Dims: {data.dims}")
        print(f"  Coords: {list(data.coords.keys())}")
        if 'height' in data.coords:
            print(f"  Height values (first 5): {data.height.values[:5]}")


🔧 Assigning height coordinates to hus composite data

📍 Processing CNTL...
   ✅ Height coordinate assigned
      New dims: ('lag', 'height', 'lat', 'lon')
      Height range: 0.01 - 22.86 km
      Shape: (9, 22, 15, 180)

📍 Processing P4K...
   ✅ Height coordinate assigned
      New dims: ('lag', 'height', 'lat', 'lon')
      Height range: 0.01 - 22.86 km
      Shape: (9, 22, 15, 180)

📍 Processing 4CO2...
   ✅ Height coordinate assigned
      New dims: ('lag', 'height', 'lat', 'lon')
      Height range: 0.01 - 22.86 km
      Shape: (9, 22, 15, 180)

✅ All experiments processed

📊 Verification:

CNTL:
  Dims: ('lag', 'height', 'lat', 'lon')
  Coords: ['height', 'lat', 'lon', 'lag']
  Height values (first 5): [22.86437528 19.53875373 17.52330287 15.84283105 13.61629278]

P4K:
  Dims: ('lag', 'height', 'lat', 'lon')
  Coords: ['height', 'lat', 'lon', 'lag']
  Height values (first 5): [22.86437528 19.53875373 17.52330287 15.84283105 13.61629278]

4CO2:
  Dims: ('lag', 'height', 'lat', 'l

## 保存hus合成数据为Dataset格式

In [ ]:
def save_hus_composite_dataset(composite_dict, save_dir=None):
    """
    将hus合成数据保存为Dataset格式，便于后续调用
    
    Parameters:
    -----------
    composite_dict : dict
        {exp: xr.DataArray} hus合成数据
    save_dir : str, optional
        保存目录，默认为 '../composite_data/'
    
    Returns:
    --------
    saved_files : dict
        {exp: filepath} 保存的文件路径
    """
    if save_dir is None:
        save_dir = '../composite_data/'
    
    os.makedirs(save_dir, exist_ok=True)
    
    print("\n" + "="*70)
    print("💾 Saving hus composite data as Dataset")
    print("="*70)
    print(f"Save directory: {save_dir}")
    
    saved_files = {}
    
    for exp in composite_dict.keys():
        print(f"\n📍 Saving {exp}...")
        
        hus_comp = composite_dict[exp]
        
        # 确保DataArray有名称
        if not hus_comp.name:
            hus_comp.name = 'hus'
        
        # 转换为Dataset
        ds = hus_comp.to_dataset()
        
        # 添加全局属性
        ds.attrs['title'] = f'Kelvin Wave Composite Analysis - Specific Humidity ({exp})'
        ds.attrs['experiment'] = exp
        ds.attrs['variable'] = 'specific_humidity'
        ds.attrs['units'] = 'kg/kg'
        ds.attrs['description'] = 'Lead-lag composite analysis of specific humidity based on detected Kelvin wave events'
        ds.attrs['lag_range'] = '-4 to +4 days'
        ds.attrs['created_date'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        # 添加维度描述
        if 'height' in ds.dims:
            ds['height'].attrs['units'] = 'km'
            ds['height'].attrs['long_name'] = 'Height above sea level'
            ds['height'].attrs['description'] = 'Converted from pressure levels using geopotential height'
        
        if 'lag' in ds.dims:
            ds['lag'].attrs['units'] = 'days'
            ds['lag'].attrs['long_name'] = 'Lag time relative to event peak'
            ds['lag'].attrs['description'] = 'Negative values = before event, positive = after event'
        
        # 设置编码以节省空间
        encoding = {
            'hus': {
                'zlib': True,
                'complevel': 4,
                'dtype': 'float32'
            }
        }
        
        # 保存文件
        filename = f'hus_composite_{exp.lower()}_with_height.nc'
        filepath = os.path.join(save_dir, filename)
        
        ds.to_netcdf(filepath, encoding=encoding)
        
        file_size = os.path.getsize(filepath) / (1024**2)
        print(f"   ✅ Saved: {filename}")
        print(f"      Size: {file_size:.2f} MB")
        print(f"      Shape: {ds['hus'].shape}")
        print(f"      Dims: {list(ds.dims.keys())}")
        
        saved_files[exp] = filepath
    
    print("\n" + "="*70)
    print(f"✅ All files saved to: {save_dir}")
    print("="*70)
    
    # 打印加载示例
    print("\n📖 To load the data later:")
    print("   ds = xr.open_dataset('../composite_data/hus_composite_cntl_with_height.nc')")
    print("   hus_composite = ds['hus']")
    
    return saved_files

# # 保存数据
saved_files = save_hus_composite_dataset(hus_composite_with_height)

# 显示保存的文件
print("\n📁 Saved files:")
for exp, filepath in saved_files.items():
    print(f"   {exp}: {filepath}")



💾 Saving wa composite data as Dataset
Save directory: ../composite_data/

📍 Saving CNTL...
   ✅ Saved: wa_composite_cntl_with_height.nc
      Size: 1.67 MB
      Shape: (9, 22, 15, 180)
      Dims: ['height', 'lat', 'lon', 'lag']

📍 Saving P4K...
   ✅ Saved: wa_composite_p4k_with_height.nc
      Size: 1.67 MB
      Shape: (9, 22, 15, 180)
      Dims: ['height', 'lat', 'lon', 'lag']

📍 Saving 4CO2...
   ✅ Saved: wa_composite_4co2_with_height.nc
      Size: 1.67 MB
      Shape: (9, 22, 15, 180)
      Dims: ['height', 'lat', 'lon', 'lag']

✅ All files saved to: ../composite_data/

📖 To load the data later:
   ds = xr.open_dataset('../composite_data/wa_composite_cntl_with_height.nc')
   wa_composite = ds['wa']

📁 Saved files:
   CNTL: ../composite_data/wa_composite_cntl_with_height.nc
   P4K: ../composite_data/wa_composite_p4k_with_height.nc
   4CO2: ../composite_data/wa_composite_4co2_with_height.nc


/tmp/ipykernel_3769932/3334985800.py:178: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"      Dims: {list(ds.dims.keys())}")


In [ ]:
def save_wa_composite_dataset(composite_dict, save_dir=None):
    """
    将wa合成数据保存为Dataset格式，便于后续调用
    
    Parameters:
    -----------
    composite_dict : dict
        {exp: xr.DataArray} wa合成数据
    save_dir : str, optional
        保存目录，默认为 '../composite_data/'
    
    Returns:
    --------
    saved_files : dict
        {exp: filepath} 保存的文件路径
    """
    if save_dir is None:
        save_dir = '../composite_data/'
    
    os.makedirs(save_dir, exist_ok=True)
    
    print("\n" + "="*70)
    print("💾 Saving wa composite data as Dataset")
    print("="*70)
    print(f"Save directory: {save_dir}")
    
    saved_files = {}
    
    for exp in composite_dict.keys():
        print(f"\n📍 Saving {exp}...")
        
        wa_comp = composite_dict[exp]
        
        # 确保DataArray有名称
        if not wa_comp.name:
            wa_comp.name = 'wa'
        
        # 转换为Dataset
        ds = wa_comp.to_dataset()
        
        # 添加全局属性
        ds.attrs['title'] = f'Kelvin Wave Composite Analysis - Vertical Velocity ({exp})'
        ds.attrs['experiment'] = exp
        ds.attrs['variable'] = 'vertical_velocity'
        ds.attrs['units'] = 'Pa/s'
        ds.attrs['description'] = 'Lead-lag composite analysis of vertical velocity based on detected Kelvin wave events'
        ds.attrs['lag_range'] = '-4 to +4 days'
        ds.attrs['created_date'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        # 添加维度描述
        if 'height' in ds.dims:
            ds['height'].attrs['units'] = 'km'
            ds['height'].attrs['long_name'] = 'Height above sea level'
            ds['height'].attrs['description'] = 'Converted from pressure levels using geopotential height'
        
        if 'lag' in ds.dims:
            ds['lag'].attrs['units'] = 'days'
            ds['lag'].attrs['long_name'] = 'Lag time relative to event peak'
            ds['lag'].attrs['description'] = 'Negative values = before event, positive = after event'

        # 设置编码以节省空间
        encoding = {
            'wa': {
                'zlib': True,
                'complevel': 4,
                'dtype': 'float32'
            }
        }
        # 保存文件
        filename = f'wa_composite_{exp.lower()}_with_height.nc'
        filepath = os.path.join(save_dir, filename)
        ds.to_netcdf(filepath, encoding=encoding)   
        file_size = os.path.getsize(filepath) / (1024**2)
        print(f"   ✅ Saved: {filename}")
        print(f"      Size: {file_size:.2f} MB")
        print(f"      Shape: {ds['wa'].shape}")
        print(f"      Dims: {list(ds.dims.keys())}")
        
        saved_files[exp] = filepath
    print("\n" + "="*70)
    print(f"✅ All files saved to: {save_dir}")
    print("="*70)

    # 打印加载示例
    print("\n📖 To load the data later:")
    print("   ds = xr.open_dataset('../composite_data/wa_composite_cntl_with_height.nc')")
    print("   wa_composite = ds['wa']")
    
    return saved_files
# 保存数据
saved_files_wa = save_wa_composite_dataset(wa_composite_with_height)
# 显示保存的文件
print("\n📁 Saved files:")
for exp, filepath in saved_files_wa.items():
    print(f"   {exp}: {filepath}")

## 验证保存的数据（可选）

In [ ]:
# 验证：重新加载并检查数据
print("="*70)
print("🔍 Verifying saved data")
print("="*70)

for exp in ['CNTL', 'P4K', '4CO2']:
    filepath = saved_files[exp]
    
    print(f"\n📍 Loading {exp} from disk...")
    
    # 重新加载
    ds = xr.open_dataset(filepath)
    hus = ds['hus']
    
    print(f"   ✅ File loaded successfully")
    print(f"   Dataset variables: {list(ds.data_vars.keys())}")
    print(f"   Dimensions: {dict(ds.dims)}")
    print(f"   Coordinates: {list(ds.coords.keys())}")
    
    if 'height' in ds.coords:
        print(f"   Height coordinate: {len(ds.height)} levels")
        print(f"   Height range: {float(ds.height.min()):.2f} - {float(ds.height.max()):.2f} km")
    
    if 'lag' in ds.coords:
        print(f"   Lag values: {ds.lag.values}")
    
    print(f"   Data shape: {hus.shape}")
    print(f"   Data range: [{float(hus.min()):.6e}, {float(hus.max()):.6e}]")
    
    # 检查属性
    if ds.attrs:
        print(f"   Global attributes: {len(ds.attrs)} items")
        print(f"      Title: {ds.attrs.get('title', 'N/A')}")
        print(f"      Created: {ds.attrs.get('created_date', 'N/A')}")

print("\n" + "="*70)
print("✅ All files verified successfully!")
print("="*70)